# Создание LanceDB (E5-base, 768d) в Google Colab

## Что нужно сделать перед запуском

### 1. Выбрать GPU-рантайм
**Runtime → Change runtime type → GPU** (T4 достаточно)

### 2. Загрузить данные в Google Drive

Создайте на Google Drive папку `thesis` (если её ещё нет) и убедитесь, что структура такая:

```
Google Drive/
└── thesis/
    └── data/
        └── posts/
            └── ajtkulov/
                └── selected/
                    └── selected500k_cleaned.jsonl
```

### 3. Важно: версия LanceDB

Перед запуском проверьте локально у себя:
```
python -c "import lancedb; print(lancedb.__version__)"
```
и впишите эту версию в переменную `LANCEDB_VERSION` в ячейке с установкой зависимостей. Иначе локально может не прочитаться формат базы.

### 4. Запустить ячейки по порядку

Готовая база будет сохранена как архив в `thesis/data/lancedb/posts3_lance.tar.gz` на Google Drive.


In [ ]:
# Установка зависимостей и монтирование Google Drive
LANCEDB_VERSION = "0.21.1"   # <-- поменяй на свою локальную версию

!pip install -q lancedb=={LANCEDB_VERSION} tantivy sentence-transformers pyarrow tqdm

from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Устройство: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Импорты, настройки и пути
import os
import json
import random
import time
import shutil
from tqdm.auto import tqdm

import numpy as np
import pyarrow as pa
import lancedb

import warnings
warnings.filterwarnings('ignore')


# Базовый путь на Google Drive
DRIVE_BASE = "/content/drive/MyDrive/thesis"

# Входной файл
selected_posts_path = os.path.join(DRIVE_BASE, "data/posts/ajtkulov/selected/selected500k_cleaned.jsonl")

# Папка для результата на Drive (сюда положим архив)
drive_output_dir = os.path.join(DRIVE_BASE, "data/lancedb")
os.makedirs(drive_output_dir, exist_ok=True)

# Локальная папка для самой базы — пишем на /content/ (быстрые I/O),
# потом архивируем и копируем готовый tar.gz на Drive
LOCAL_LANCEDB_PATH = "/content/lancedb_store"
TABLE_NAME         = "e5-base-base-50k"

# Проверяем, что входной файл на месте
assert os.path.exists(selected_posts_path), f"Файл не найден: {selected_posts_path}"
print(f"Входной файл: {selected_posts_path}")
print(f"  размер: {os.path.getsize(selected_posts_path) / 1e6:.1f} MB")
print(f"Папка для результата: {drive_output_dir}")


In [ ]:
# Параметры модели и векторизации
BI_ENCODER_PATH = "intfloat/multilingual-e5-base"   # 768d, из коробки
MAX_POSTS       = 50_000
BATCH_SIZE      = 384      # T4 спокойно тянет; V100 — уменьши до 256
RANDOM_SEED     = 42
CLEAN_CATEGORY  = True
USE_E5_PREFIX   = True     # E5 требует passage:/query: префиксы
DEVICE          = "cuda"


In [ ]:
# Загрузка модели
from sentence_transformers import SentenceTransformer

bi_encoder = SentenceTransformer(BI_ENCODER_PATH, device=DEVICE)
EMBEDDING_DIM = bi_encoder.get_sentence_embedding_dimension()
print(f"Модель: {BI_ENCODER_PATH}")
print(f"  dim: {EMBEDDING_DIM}")
print(f"  max_seq_len: {bi_encoder.max_seq_length}")


In [ ]:
# Чтение и семплирование постов
all_posts = []
with open(selected_posts_path, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Чтение постов"):
        obj = json.loads(line)
        if not obj.get('text', '').strip():
            continue
        if CLEAN_CATEGORY:
            cat = obj.get('category', '')
            if '|||' in cat:
                obj['category'] = cat.split('|||')[0].strip()
        all_posts.append(obj)

print(f"Всего постов в файле: {len(all_posts):,}")

rng = random.Random(RANDOM_SEED)
posts = rng.sample(all_posts, MAX_POSTS)
print(f"Случайная выборка (seed={RANDOM_SEED}): {len(posts):,}")


In [ ]:
# Создание локальной LanceDB
db = lancedb.connect(LOCAL_LANCEDB_PATH)

schema = pa.schema([
    pa.field("vector", pa.list_(pa.float32(), EMBEDDING_DIM)),
    pa.field("text", pa.utf8()),
    pa.field("channel", pa.utf8()),
    pa.field("category", pa.utf8()),
    pa.field("post_id", pa.utf8()),
    pa.field("link", pa.utf8()),
    pa.field("date", pa.utf8()),
    pa.field("views", pa.utf8()),
])

if TABLE_NAME in db.table_names():
    db.drop_table(TABLE_NAME)
    print(f"Старая таблица {TABLE_NAME} удалена")

table = db.create_table(TABLE_NAME, schema=schema)
print(f"Создана таблица {TABLE_NAME} в {LOCAL_LANCEDB_PATH}")


In [ ]:
# Векторизация и запись
def make_post_id(post):
    return f"{post['channel']}::{post['id']}"

total = len(posts)
print(f"Старт: {total:,} постов, batch={BATCH_SIZE}, device={DEVICE}")

t_start = time.time()
indexed = 0
pbar = tqdm(total=total, desc="Векторизация", unit="пост")

for batch_start in range(0, total, BATCH_SIZE):
    batch = posts[batch_start : batch_start + BATCH_SIZE]
    texts = [p['text'] for p in batch]

    # E5: префикс 'passage: ' для документов
    encoded_texts = ["passage: " + t for t in texts] if USE_E5_PREFIX else texts

    embeddings = bi_encoder.encode(
        encoded_texts,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=BATCH_SIZE,
        device=DEVICE,
    )

    records = [{
        "vector": embeddings[i].tolist(),
        "text": batch[i]['text'],             # без префикса
        "channel": batch[i]['channel'],
        "category": batch[i].get('category', ''),
        "post_id": make_post_id(batch[i]),
        "link": batch[i].get('link', ''),
        "date": batch[i].get('date', ''),
        "views": str(batch[i].get('views', '')),
    } for i in range(len(batch))]

    table.add(records)
    indexed += len(records)

    elapsed = time.time() - t_start
    speed = indexed / elapsed
    eta = (total - indexed) / speed if speed > 0 else 0
    pbar.update(len(records))
    pbar.set_postfix({"скорость": f"{speed:.0f} п/с", "ETA": f"{eta/60:.1f} мин"})

pbar.close()
print(f"\nГотово: {indexed:,} постов за {time.time()-t_start:.0f}с")
print(f"В таблице: {table.count_rows():,} записей")


In [ ]:
# FTS-индекс (BM25)
print("Создание FTS-индекса на поле 'text'...")
table.create_fts_index("text", replace=True)
print("Готово")


In [ ]:
# Санити-чек: тестовый запрос
test_query = "Кроссовки Nike Air Max мужские для бега, размер 42, чёрные"
print(f"Запрос: {test_query}\n")

q_for_enc = ("query: " + test_query) if USE_E5_PREFIX else test_query
qvec = bi_encoder.encode([q_for_enc], normalize_embeddings=True)[0].tolist()

print("=== ВЕКТОРНЫЙ ПОИСК ===")
for i, r in enumerate(
    table.search(qvec, query_type="vector").limit(5).select(["text", "channel", "category"]).to_list(), 1
):
    print(f"  {i}. [{r['category']}] @{r['channel']}")
    print(f"     {r['text'][:120]}...")
    print()

print("=== BM25 ===")
for i, r in enumerate(
    table.search(test_query, query_type="fts").limit(5).select(["text", "channel", "category"]).to_list(), 1
):
    print(f"  {i}. [{r['category']}] @{r['channel']}")
    print(f"     {r['text'][:120]}...")
    print()


In [ ]:
# Статистика
def dir_size_mb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024 * 1024)

total_rows = table.count_rows()
db_size = dir_size_mb(LOCAL_LANCEDB_PATH)

print(f"Записей: {total_rows:,}")
print(f"Размер папки {LOCAL_LANCEDB_PATH}: {db_size:.1f} MB")
print(f"Среднее на запись: {db_size * 1024 / total_rows:.1f} KB")


In [ ]:
# Архивирование папки posts3.lance и копирование на Drive
ARCHIVE_NAME = "posts3_lance.tar.gz"
local_archive = f"/content/{ARCHIVE_NAME}"

# Убедимся, что таблица лежит на диске
table_dir = os.path.join(LOCAL_LANCEDB_PATH, f"{TABLE_NAME}.lance")
assert os.path.exists(table_dir), f"Таблица не найдена на диске: {table_dir}"

# Архивируем папку таблицы целиком
!tar -czf {local_archive} -C {LOCAL_LANCEDB_PATH} {TABLE_NAME}.lance

archive_size_mb = os.path.getsize(local_archive) / 1e6
print(f"Архив создан: {local_archive} ({archive_size_mb:.1f} MB)")

# Копируем на Google Drive
drive_archive = os.path.join(drive_output_dir, ARCHIVE_NAME)
shutil.copy(local_archive, drive_archive)

assert os.path.exists(drive_archive), "Не удалось скопировать архив на Drive"
print(f"✓ Архив сохранён на Drive: {drive_archive}")
print(f"  Размер: {os.path.getsize(drive_archive) / 1e6:.1f} MB")


## Что делать локально после скачивания

С Google Drive забираем `thesis/data/lancedb/posts3_lance.tar.gz` и раскладываем рядом с существующими таблицами `posts` / `posts2`:

```bash
cd thesis/
tar -xzf /path/to/posts3_lance.tar.gz -C lancedb_store/
```

Появится `thesis/lancedb_store/posts3.lance/`. После этого можно запускать `benchmark-pipeline.ipynb` — там уже прописано `TABLE_E5 = "e5-base-base-50k"`.

Если локально уже есть старый `posts3.lance` — удали его руками и повтори распаковку.
